In [22]:
import numpy as np

PIE_PATH     = '/content/drive/MyDrive/PIE'
FEATURES_PATH = f'{PIE_PATH}/features'

X_data = np.load(f'{FEATURES_PATH}/X_sequences.npy')
y_data = np.load(f'{FEATURES_PATH}/y_labels.npy')

print(f'X_data shape: {X_data.shape}')
print(f'y_data shape: {y_data.shape}')
print(f'Unique labels: {np.unique(y_data)}')
print(f'Crossing samples: {np.isin(y_data, [2,4,5]).sum()}')
print(f'Non-crossing:     {np.isin(y_data, [0,1,3]).sum()}')

X_data shape: (3888, 30, 36)
y_data shape: (82791,)
Unique labels: [0 1 2 3 4 5]
Crossing samples: 33951
Non-crossing:     48840


In [23]:
F_CURRENT_SPEED        = 0
F_ACCELERATION         = 3
F_SPEED_VARIANCE       = 5
F_FORWARD_LEAN         = 18
F_HESITATION_CYCLES    = 23
F_DISTANCE_CHANGE_RATE = 26
F_VEHICLE_DISTANCE     = 32

In [24]:
def compute_empirical_weights(X_data, y_data):
    from scipy.stats import pointbiserialr

    # The y_data has more samples (82791) than X_data has sequences (3888).
    # We assume y_data needs to be truncated to match X_data's first dimension
    # to align the labels with the corresponding sequences.
    y_binary = np.isin(y_data[:X_data.shape[0]], [2, 4, 5]).astype(int)
    X_last   = X_data[:, -1, :]

    feature_indices = {
        'p_distance': F_VEHICLE_DISTANCE,
        'p_toward':   F_DISTANCE_CHANGE_RATE,
        'p_speed':    F_CURRENT_SPEED,
        'p_accel':    F_ACCELERATION,
        'p_lean':     F_FORWARD_LEAN,
        'p_variance': F_SPEED_VARIANCE,
        'p_hesit':    F_HESITATION_CYCLES,
    }

    correlations = {}
    for name, idx in feature_indices.items():
        r, _ = pointbiserialr(y_binary, X_last[:, idx])
        correlations[name] = abs(r)

    total   = sum(correlations.values())
    weights = {k: v / total for k, v in correlations.items()}

    print('\n── Empirical weights from PIE data ──')
    for name, w in sorted(weights.items(), key=lambda x: -x[1]):
        print(f'  {name:<12} correlation={correlations[name]:.4f}  weight={w:.4f}')
    print(f'  Total = {sum(weights.values()):.4f}')
    return weights

# ← THIS LINE IS WHAT WAS MISSING
weights = compute_empirical_weights(X_data, y_data)


── Empirical weights from PIE data ──
  p_distance   correlation=0.0753  weight=0.3270
  p_variance   correlation=0.0539  weight=0.2340
  p_lean       correlation=0.0313  weight=0.1359
  p_hesit      correlation=0.0272  weight=0.1182
  p_accel      correlation=0.0263  weight=0.1144
  p_toward     correlation=0.0093  weight=0.0403
  p_speed      correlation=0.0070  weight=0.0302
  Total = 1.0000


In [25]:
import json
import numpy as np

FEATURES_PATH = '/content/drive/MyDrive/PIE/features'

# Load the scaler stats
with open(f'{FEATURES_PATH}/feature_stats.json') as f:
    stats = json.load(f)

print('feature_stats.json keys:', list(stats.keys()))
print('\nFull content:')
print(json.dumps(stats, indent=2))

feature_stats.json keys: ['total_frames', 'total_sequences', 'seq_len', 'seq_step', 'feature_dim', 'feature_names', 'label_names', 'class_counts_frames', 'class_counts_sequences', 'split_sizes', 'feature_means', 'feature_stds']

Full content:
{
  "total_frames": 82791,
  "total_sequences": 3888,
  "seq_len": 30,
  "seq_step": 15,
  "feature_dim": 36,
  "feature_names": [
    "current_speed",
    "avg_speed",
    "max_speed",
    "acceleration",
    "deceleration",
    "speed_variance",
    "step_frequency",
    "left_step_length",
    "right_step_length",
    "pause_between_steps",
    "upper_body_angle",
    "lower_body_angle",
    "head_angle",
    "head_turn_frequency",
    "shoulder_angle",
    "hip_angle",
    "foot_angle_left",
    "foot_angle_right",
    "forward_lean",
    "lateral_lean",
    "body_orientation",
    "body_orientation_change",
    "pause_duration",
    "hesitation_cycles",
    "total_hesitation_time",
    "distance_to_curb",
    "distance_change_rate",
    "temp

In [26]:
import joblib
import numpy as np

FEATURES_PATH = '/content/drive/MyDrive/PIE/features'

# Load scaler
scaler = joblib.load(f'{FEATURES_PATH}/feature_scaler.pkl')

# Load test data (currently scaled)
X_test = np.load(f'{FEATURES_PATH}/X_test.npy')
y_test = np.load(f'{FEATURES_PATH}/y_test.npy')

# Unscale — reshape to 2D, inverse transform, reshape back
n, seq, feat = X_test.shape
X_test_raw = scaler.inverse_transform(
    X_test.reshape(-1, feat)
).reshape(n, seq, feat)

# Verify — these should now match feature_stats means
X_last = X_test_raw[:, -1, :]
print(f'vehicle_distance mean : {X_last[:, 32].mean():.4f}  (expect ≈0.3391)')
print(f'speed mean            : {X_last[:,  0].mean():.4f}  (expect ≈6.6076)')
print(f'forward_lean mean     : {X_last[:, 18].mean():.4f}  (expect ≈20.4451)')
print(f'speed_variance mean   : {X_last[:,  5].mean():.4f}  (expect ≈134.815)')

vehicle_distance mean : 0.3403  (expect ≈0.3391)
speed mean            : 6.3430  (expect ≈6.6076)
forward_lean mean     : 16.4367  (expect ≈20.4451)
speed_variance mean   : 119.7367  (expect ≈134.815)
